# 로컬 원본부터 전처리
7개 입력은 data/raw_input에 준비. 데이터를 다운로드/업로드하지 않는다.

In [ ]:
from pathlib import Path
import os,sys,nbformat
from nbclient import NotebookClient
root=next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'notebooks/157_rg3_existing_connected_pipeline_runner.ipynb').exists())
rawdir=root/'data/raw_input'
required=['moldset_labeled.csv','labeled_data.csv','unlabeled_data.csv','moldset_labeled_cn7.csv','moldset_labeled_rg3.csv','moldset_unlabeled_cn7.csv','moldset_unlabeled_rg3.csv']
missing=[x for x in required if not (rawdir/x).is_file()]
if missing:raise FileNotFoundError('Place these LOCAL files in data/raw_input: '+str(missing))
(root/'reports/tables').mkdir(exist_ok=True,parents=True)
for filename in ['09_original_product_splits.ipynb','10_original_defect_report_revision.ipynb']:
 n=nbformat.read(root/'notebooks'/filename,as_version=4)
 for c in n.cells:
  if c.cell_type=='code':
   c.source=c.source.replace("source = root / 'data/original'","source = root / 'data/raw_input'").replace("root / 'data/moldset_labeled_cn7.csv'","root / 'data/raw_input/moldset_labeled_cn7.csv'")
 print('Preprocess',filename,flush=True)
 NotebookClient(n,timeout=1200,kernel_name='python3',resources={'metadata':{'path':str(root)}}).execute()
for filename in ['01_link_primary_secondary.ipynb','00_base_labeled.ipynb']:
 p=root/'notebooks/preprocessing'/filename;n=nbformat.read(p,as_version=4)
 print('Preprocess',filename,flush=True)
 NotebookClient(n,timeout=1200,kernel_name='python3',resources={'metadata':{'path':str(root)}}).execute();nbformat.write(n,p)
td=root/'data/taein';sys.path.insert(0,str(td))
from final_model import train,test
previous=Path.cwd()
try:
 os.chdir(td);train(products=('rg3',));test(products=('rg3',))
 import semi_normal
 loaded=semi_normal.load_std('rg3')
finally:os.chdir(previous)
lab=loaded['lab'][loaded['lab']['eval']]
assert len(lab)==591 and lab.type.value_counts().to_dict()=={'정상':566,'가스':17,'미성형':8}
print('Preprocessing ready: run159 then157.')
